Prévision du chiffre d'affaires quotidien — Machine Learning
Objectif : développer et évaluer un modèle capable de prédire le chiffre d'affaires net du lendemain pour chacun des trois établissements à partir des données historiques disponibles.

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

In [60]:
performance_quotidienne = pd.read_csv('../data/processed/performance_quotidienne.csv', parse_dates=['Date'])

performance_quotidienne = performance_quotidienne.sort_values(['Outlet', 'Date']).reset_index(drop=True)

performance_quotidienne.head()

,Date,Outlet,CA_net,Nb_transactions,Nb_articles,Panier_moyen,Jour_semaine,Mois
0,2025-01-02,SHOP001,1808108.121,82,94,22050.099037,Thursday,1
1,2025-01-03,SHOP001,2334234.248,101,122,23111.230178,Friday,1
2,2025-01-04,SHOP001,2227927.944,99,115,22504.322667,Saturday,1
3,2025-01-05,SHOP001,1904504.519,90,106,21161.161322,Sunday,1
4,2025-01-06,SHOP001,1698198.207,82,90,20709.734232,Monday,1


In [61]:
#vérification de la continuité de dates dans le dataset

couverture = performance_quotidienne.groupby('Outlet')['Date'].agg(['min', 'max', 'nunique'])

couverture['Jours_attendus'] = (couverture['max'] - couverture['min']).dt.days + 1

couverture['Jours_manquants'] = couverture['Jours_attendus'] - couverture['nunique']

couverture

#Je n'ai aucune date manquante dans mes observations 

,min,max,nunique,Jours_attendus,Jours_manquants
Outlet,,,,,
SHOP001,2025-01-02,2025-09-30,272,272,0
SHOP002,2025-01-02,2025-09-30,272,272,0
SHOP003,2025-01-02,2025-09-30,272,272,0


In [62]:
# verification de l'autocorrélation des CA: Il s'agit de déterminer si le CA de la veille à une relation
# avec le CA d'aujourd'hui. Et éventuellement rechercher les causes de ces corrélations. 
performance_ml = performance_quotidienne.copy()

decalages = [1, 7, 14]

autocorrelations = pd.DataFrame({
    outlet: [
        groupe['CA_net'].autocorr(lag=lag)
        for lag in decalages
    ]
    for outlet, groupe in performance_ml.groupby('Outlet')
}, index=decalages)

autocorrelations.index.name = 'Décalage (jours)'

autocorrelations

,SHOP001,SHOP002,SHOP003
Décalage (jours),,,
1,0.503120,0.608533,0.620648
7,0.308275,0.550983,0.452582
14,0.235453,0.473926,0.381649


Il existe une relation avec le CA de la veille et le CA du jour J dans les 3 SHOP. L'autocorrélation diminue avec le temps. Cela suggére que les observations récentes contiennent plus d'informations que celles plus anciennes.
Maintenant que nous avons vu qu'il existe une relation statistique entre nos CA, Notre objectif est de prédire le CA du jour J à partir des informations disponibles à la clôture de J−1.
La variable cible est 'CA_net'. Nous allons construire les variables explicatives historiques à partir des CA des journées précédentes, sans utiliser d'informations du jour à prédire.

In [63]:
performance_ml = performance_quotidienne.copy()

In [64]:
performance_ml.head(10)

,Date,Outlet,CA_net,Nb_transactions,Nb_articles,Panier_moyen,Jour_semaine,Mois
0,2025-01-02,SHOP001,1808108.121,82,94,22050.099037,Thursday,1
1,2025-01-03,SHOP001,2334234.248,101,122,23111.230178,Friday,1
2,2025-01-04,SHOP001,2227927.944,99,115,22504.322667,Saturday,1
3,2025-01-05,SHOP001,1904504.519,90,106,21161.161322,Sunday,1
4,2025-01-06,SHOP001,1698198.207,82,90,20709.734232,Monday,1
5,2025-01-07,SHOP001,2227027.039,96,111,23198.198323,Tuesday,1
6,2025-01-08,SHOP001,2383783.798,110,125,21670.761800,Wednesday,1
7,2025-01-09,SHOP001,1764864.875,72,85,24512.012153,Thursday,1
8,2025-01-10,SHOP001,1903603.616,90,103,21151.151289,Friday,1
9,2025-01-11,SHOP001,1739639.652,77,90,22592.722753,Saturday,1


Découpage chronologique et construction des variables
La frontière entraînement / test est fixée au 1er août 2025 : les journées du 2 janvier au 31 juillet servent à l'entraînement et à la validation, celles du 1er août au 30 septembre au test final.
Nous construisons d'abord les variables historiques sur la série complète, uniquement à partir des jours antérieurs à la date prédite. Nous appliquons ensuite la frontière chronologique : ainsi, la prévision du 1er août peut utiliser le CA du 31 juillet, sans fuite temporelle. Les paramètres des prétraitements et des modèles seront appris exclusivement sur les données d'entraînement.

In [65]:
performance_ml = performance_ml.sort_values(['Date', 'Outlet']).reset_index(drop=True)

train = performance_ml.loc[performance_ml['Date'] < '2025-08-01'].copy()
test = performance_ml.loc[performance_ml['Date'] >= '2025-08-01'].copy()

print('Train :', train.shape)
print('Test :', test.shape)

Train : (633, 8)
Test : (183, 8)


In [66]:
print('Train :', train['Date'].min(), '→', train['Date'].max())
print('Test :', test['Date'].min(), '→', test['Date'].max())

Train : 2025-01-02 00:00:00 → 2025-07-31 00:00:00
Test : 2025-08-01 00:00:00 → 2025-09-30 00:00:00


In [67]:
# Variable cible et variables explicatives

performance_ml = performance_ml.sort_values(['Outlet', 'Date']).reset_index(drop=True)

performance_ml['CA_J_1'] = performance_ml.groupby('Outlet')['CA_net'].shift(1) #CA de la veille

performance_ml['CA_J_7'] = performance_ml.groupby('Outlet')['CA_net'].shift(7) #CA net des SHOP
# observé 7 jours auparavant

performance_ml['CA_moyen_7j'] = performance_ml.groupby('Outlet')['CA_net'].transform(
    lambda x: x.shift(1).rolling(window=7).mean()) # moyenne glissante du CA observé sur 7 jours dans
    # différents SHOP

In [68]:
performance_ml[['Date', 'Outlet', 'CA_net', 'CA_J_1', 'CA_J_7', 'CA_moyen_7j']].iloc[7:12]

,Date,Outlet,CA_net,CA_J_1,CA_J_7,CA_moyen_7j
7,2025-01-09,SHOP001,1764864.875,2383783.798,1808108.121,2.083398e+06
8,2025-01-10,SHOP001,1903603.616,1764864.875,2334234.248,2.077220e+06
9,2025-01-11,SHOP001,1739639.652,1903603.616,2227927.944,2.015701e+06
10,2025-01-12,SHOP001,2130630.642,1739639.652,1904504.519,1.945946e+06
11,2025-01-13,SHOP001,1718918.929,2130630.642,1698198.207,1.978250e+06


In [69]:
train = performance_ml.loc[performance_ml['Date'] < '2025-08-01'].copy()
test = performance_ml.loc[performance_ml['Date'] >= '2025-08-01'].copy()

In [70]:
variables_historiques = ['CA_J_1', 'CA_J_7', 'CA_moyen_7j']

train = train.dropna(subset=variables_historiques)
test = test.dropna(subset=variables_historiques)

print('Train :', train.shape)
print('Test :', test.shape)

Train : (612, 11)
Test : (183, 11)


In [71]:
train_model = train.loc[train['Date'] < '2025-07-01'].copy()
validation = train.loc[train['Date'] >= '2025-07-01'].copy()

print('Train model :', train_model.shape)
print('Validation :', validation.shape)

Train model : (519, 11)
Validation : (93, 11)


In [72]:
from sklearn.metrics import mean_absolute_error

y_reel = validation['CA_net']
y_prevu = validation['CA_J_1']

mae = mean_absolute_error(y_reel, y_prevu)

print('Erreur moyenne :', round(mae, 2), 'IDR')

Erreur moyenne : 244851.3 IDR


In [73]:
mae_par_shop = validation.groupby('Outlet').apply(
    lambda groupe: mean_absolute_error(groupe['CA_net'], groupe['CA_J_1']),
    include_groups=False
)

mae_par_shop

Outlet
SHOP001    276721.885258
SHOP002    179424.586839
SHOP003    278407.441194
dtype: float64

In [74]:
ca_moyen_validation = validation.groupby('Outlet')['CA_net'].mean()

erreur_relative = (mae_par_shop / ca_moyen_validation) * 100

erreur_relative

Outlet
SHOP001    11.867343
SHOP002    14.410083
SHOP003    18.006165
dtype: float64

J'ai évalué une première méthode de prévision naïve consistant à prédire le chiffre d'affaires du jour J à partir de celui de la veille (J−1).

La période de validation s'étend du 1er au 31 juillet 2025, soit 31 journées par établissement et 93 observations au total. Les données de janvier à juin sont réservées à l'entraînement, tandis que celles d'août et septembre constituent notre jeu de test final.

Sur cette période de validation, la méthode présente une erreur absolue moyenne (MAE) de 244 851 IDR, tous établissements confondus.

L'analyse par établissement révèle des performances différentes :

SHOP001 : 276 722 IDR, soit 11,87 % de son CA quotidien moyen.

SHOP002 : 179 425 IDR, soit 14,41 %.

SHOP003 : 278 407 IDR, soit 18,01 %.

SHOP001 présente la meilleure précision relative, tandis que SHOP003 enregistre les erreurs relatives les plus importantes.

Ces résultats montrent que la pertinence d'une prévision fondée uniquement sur le CA de la veille varie selon les établissements.

Cette première baseline constitue notre référence pour déterminer si une moyenne mobile ou un modèle de Machine Learning permet d'améliorer la précision des prévisions.

In [75]:
mae_moyenne_7j = validation.groupby('Outlet').apply(
    lambda groupe: mean_absolute_error(groupe['CA_net'], groupe['CA_moyen_7j']),
    include_groups=False
)

comparaison_baselines = pd.DataFrame({
    'CA_veille': mae_par_shop,
    'Moyenne_7j': mae_moyenne_7j
})

comparaison_baselines

,CA_veille,Moyenne_7j
Outlet,,
SHOP001,276721.885258,234325.571419
SHOP002,179424.586839,135666.542028
SHOP003,278407.441194,191676.008594


Sur la période de validation du **1er au 31 juillet 2025**, la prévision basée sur la moyenne du CA des sept journées précédentes surpasse celle utilisant uniquement le CA de la veille, pour les trois établissements.

Les réductions de MAE atteignent 15,4 % pour SHOP001, 24,4 % pour SHOP002 et 31,2 % pour SHOP003.

Ces résultats suggèrent que le niveau d'activité moyen récent constitue une référence prédictive plus pertinente que la dernière journée observée.

**La moyenne mobile sur sept jours est donc retenue comme baseline principale** pour nos prochaines comparaisons avec les modèles de Machine Learning.

Cette conclusion reste provisoire et devra être confirmée lors de l'évaluation finale sur août et septembre 2025.

In [76]:
variables_ml = ['CA_J_1', 'CA_J_7', 'CA_moyen_7j', 'Jour_semaine', 'Mois', 'Outlet']

X_train = train_model[variables_ml].copy()
y_train = train_model['CA_net'].copy()

X_train.head()

,CA_J_1,CA_J_7,CA_moyen_7j,Jour_semaine,Mois,Outlet
7,2383783.798,1808108.121,2.083398e+06,Thursday,1,SHOP001
8,1764864.875,2334234.248,2.077220e+06,Friday,1,SHOP001
9,1903603.616,2227927.944,2.015701e+06,Saturday,1,SHOP001
10,1739639.652,1904504.519,1.945946e+06,Sunday,1,SHOP001
11,2130630.642,1698198.207,1.978250e+06,Monday,1,SHOP001


In [77]:
X_val = validation[variables_ml].copy()
y_val = validation['CA_net'].copy()

print('X_train :', X_train.shape)
print('y_train :', y_train.shape)

print('X_val :', X_val.shape)
print('y_val :', y_val.shape)

X_train : (519, 6)
y_train : (519,)
X_val : (93, 6)
y_val : (93,)


In [78]:
# Pré-traitement de données

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

variables_numeriques = ['CA_J_1', 'CA_J_7', 'CA_moyen_7j']
variables_categorielles = ['Jour_semaine', 'Mois', 'Outlet']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), variables_numeriques), #standardisation des variables numériques
    ('cat', OneHotEncoder(handle_unknown='ignore'), variables_categorielles) #encodage des variables
    #catégorielles
])

In [79]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

modele_lineaire = Pipeline([('preprocessing', preprocessor), ('regression', LinearRegression())])

In [80]:
modele_lineaire.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['CA_J_1','CA_J_7','CA_moyen_7j','Jour_semaine','Mois','Outlet']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis 

In [81]:
y_pred_lineaire = modele_lineaire.predict(X_val)

y_pred_lineaire[:5]

array([2441807.84071551, 2381264.76017429, 2352008.52013808,
       2263394.12964166, 2153495.34301242])

In [82]:
mae_lineaire = mean_absolute_error(y_val, y_pred_lineaire)

print('MAE régression linéaire :', round(mae_lineaire, 2), 'IDR')

MAE régression linéaire : 214808.82 IDR


In [83]:
mae_baseline_7j = mean_absolute_error(y_val, validation['CA_moyen_7j'])

print('MAE baseline 7 jours :', round(mae_baseline_7j, 2), 'IDR')
print('MAE régression linéaire :', round(mae_lineaire, 2), 'IDR')

MAE baseline 7 jours : 187222.71 IDR
MAE régression linéaire : 214808.82 IDR


In [84]:
comparaison_modeles = validation[['Outlet', 'CA_net', 'CA_moyen_7j']].copy()
comparaison_modeles['Prediction_lineaire'] = y_pred_lineaire

mae_lineaire_shop = comparaison_modeles.groupby('Outlet').apply(
    lambda groupe: mean_absolute_error(groupe['CA_net'], groupe['Prediction_lineaire']),
    include_groups=False
)

comparaison_par_shop = pd.DataFrame({
    'Baseline_7j': mae_moyenne_7j,
    'Regression_lineaire': mae_lineaire_shop
})

comparaison_par_shop

,Baseline_7j,Regression_lineaire
Outlet,,
SHOP001,234325.571419,206012.118867
SHOP002,135666.542028,227028.424682
SHOP003,191676.008594,211385.920600


Sur la période de validation du **1er au 31 juillet 2025**, la régression linéaire présente une MAE globale de **214 809 IDR**, contre **187 223 IDR** pour la baseline fondée sur la moyenne mobile de sept jours.

L'analyse par établissement révèle des performances contrastées :

- **SHOP001 :** amélioration de 12 % par rapport à la baseline.
- **SHOP002 :** dégradation de 67,3 %.
- **SHOP003 :** dégradation de 10,3 %.

La régression linéaire n'améliore donc pas globalement les prévisions. Ses performances varient sensiblement selon les établissements.

**Une limite possible concerne la variable 'Mois' :** le modèle a été entraîné sur janvier à juin, tandis que juillet constitue une catégorie inconnue pour l'encodeur One-Hot.

Nous allons donc entraîner une nouvelle régression sans cette variable afin de vérifier si elle contribue à la dégradation des performances.

Cette correction constitue un test méthodologique : elle ne garantit pas une amélioration et ses résultats devront être comparés à ceux du modèle initial et de la baseline.

In [85]:
from sklearn.base import clone

variables_ml_sans_mois = ['CA_J_1', 'CA_J_7', 'CA_moyen_7j', 'Jour_semaine', 'Outlet']

X_train_sans_mois = train_model[variables_ml_sans_mois]
X_val_sans_mois = validation[variables_ml_sans_mois]

preprocessor_sans_mois = ColumnTransformer([
    ('num', StandardScaler(), ['CA_J_1', 'CA_J_7', 'CA_moyen_7j']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['Jour_semaine', 'Outlet'])
])

modele_sans_mois = Pipeline([
    ('preprocessing', preprocessor_sans_mois),
    ('regression', LinearRegression())
])

modele_sans_mois.fit(X_train_sans_mois, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['CA_J_1','CA_J_7','CA_moyen_7j','Jour_semaine','Outlet']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concate

In [86]:
y_pred_sans_mois = modele_sans_mois.predict(X_val_sans_mois)

mae_sans_mois = mean_absolute_error(y_val, y_pred_sans_mois)

print('Baseline 7j :', round(mae_baseline_7j, 2))
print('Régression initiale :', round(mae_lineaire, 2))
print('Régression sans mois :', round(mae_sans_mois, 2))

Baseline 7j : 187222.71
Régression initiale : 214808.82
Régression sans mois : 189013.51


In [87]:
comparaison_modeles['Prediction_sans_mois'] = y_pred_sans_mois

mae_sans_mois_shop = comparaison_modeles.groupby('Outlet').apply(
    lambda groupe: mean_absolute_error(groupe['CA_net'], groupe['Prediction_sans_mois']),
    include_groups=False
)

comparaison_finale = pd.DataFrame({
    'Baseline_7j': mae_moyenne_7j,
    'Regression_initiale': mae_lineaire_shop,
    'Regression_sans_mois': mae_sans_mois_shop
})

comparaison_finale

,Baseline_7j,Regression_initiale,Regression_sans_mois
Outlet,,,
SHOP001,234325.571419,206012.118867,214469.709124
SHOP002,135666.542028,227028.424682,146735.020442
SHOP003,191676.008594,211385.920600,205835.803026


Sur la période de validation de juillet 2025, la suppression de la variable 'Mois' réduit la MAE globale de la régression linéaire de 214 809 à 189 014 IDR, soit une amélioration d'environ 12 %.

Cette amélioration est principalement portée par SHOP002. Pour SHOP003, le gain est plus modéré, tandis que les performances de SHOP001 se dégradent légèrement.

Malgré cette correction, **la baseline fondée sur la moyenne mobile de sept jours conserve la meilleure performance globale**, avec une MAE de 187 223 IDR.

Ces résultats suggèrent que la représentation des variables calendaires influence les performances du modèle, sans démontrer que leur suppression sera systématiquement bénéfique sur d'autres périodes.

In [88]:
from sklearn.linear_model import Ridge
from sklearn.base import clone

modele_ridge = Pipeline([
    ('preprocessing', clone(preprocessor_sans_mois)),
    ('regression', Ridge(alpha=1.0))
])

modele_ridge.fit(X_train_sans_mois, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['CA_J_1','CA_J_7','CA_moyen_7j','Jour_semaine','Outlet']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concate

In [89]:
y_pred_ridge = modele_ridge.predict(X_val_sans_mois)

mae_ridge = mean_absolute_error(y_val, y_pred_ridge)

comparaison_mae = pd.Series({
    'Baseline 7j': mae_baseline_7j,
    'Régression initiale': mae_lineaire,
    'Régression sans mois': mae_sans_mois,
    'Ridge': mae_ridge
})

comparaison_mae.round(2)

Baseline 7j             187222.71
Régression initiale     214808.82
Régression sans mois    189013.51
Ridge                   189582.05
dtype: float64

Sur la période de validation du **1er au 31 juillet 2025**, Ridge ('alpha=1') obtient une MAE de **189 582 IDR**, contre **189 014 IDR** pour la régression linéaire sans régularisation.

L'introduction de la régularisation L2 n'améliore donc pas les performances dans cette configuration.

La moyenne mobile sur sept jours conserve la meilleure précision globale, avec une MAE de **187 223 IDR**.

À ce stade, la complexification du modèle n'apporte pas de gain prédictif démontré. Les résultats restent toutefois limités à une seule période de validation.

In [90]:
pred_train_lineaire = modele_sans_mois.predict(X_train_sans_mois)
pred_train_ridge = modele_ridge.predict(X_train_sans_mois)

mae_train_lineaire = mean_absolute_error(y_train, pred_train_lineaire)
mae_train_ridge = mean_absolute_error(y_train, pred_train_ridge)

In [91]:
comparaison_generalisation = pd.DataFrame({
    'MAE_train': [mae_train_lineaire, mae_train_ridge],
    'MAE_validation': [mae_sans_mois, mae_ridge]
}, index=['Regression_lineaire', 'Ridge'])

comparaison_generalisation['Ecart_MAE'] = comparaison_generalisation['MAE_validation'] - comparaison_generalisation['MAE_train']

comparaison_generalisation.round(2)

,MAE_train,MAE_validation,Ecart_MAE
Regression_lineaire,181716.82,189013.51,7296.69
Ridge,181896.07,189582.05,7685.98


Sur les périodes d'entraînement (janvier–juin 2025) et de validation (juillet 2025), les régressions linéaire et Ridge présentent des erreurs relativement proches, avec des écarts de MAE d'environ 4 %.

Ces résultats ne révèlent pas de signe marqué de surapprentissage, sans pour autant démontrer son absence.

La régularisation Ridge (alpha=1) n'améliore ni l'erreur d'entraînement ni celle de validation.

La moyenne mobile sur sept jours conserve la meilleure performance sur juillet. À ce stade, **la complexification du modèle linéaire n'apporte pas de gain prédictif démontré**.

Les régressions linéaire et Ridge ne dépassent pas la moyenne mobile de sept jours sur juillet. Nous testons donc Random Forest pour déterminer si la prise en compte de relations non linéaires et d'interactions entre variables améliore les prévisions.
Pour isoler l'effet du changement de modèle, nous conservons les mêmes variables explicatives que dans la régression sans Mois, ainsi que les mêmes périodes d'entraînement et de validation. La complexité des arbres est limitée (max_depth=5, min_samples_leaf=5) afin de réduire le risque de surapprentissage.

In [92]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone

modele_rf = Pipeline([
    ('preprocessing', clone(preprocessor_sans_mois)),
    ('regression', RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=5, random_state=42))
])

modele_rf.fit(X_train_sans_mois, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['CA_J_1','CA_J_7','CA_moyen_7j','Jour_semaine','Outlet']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concate

In [93]:
pred_train_rf = modele_rf.predict(X_train_sans_mois)
pred_val_rf = modele_rf.predict(X_val_sans_mois)

mae_train_rf = mean_absolute_error(y_train, pred_train_rf)
mae_val_rf = mean_absolute_error(y_val, pred_val_rf)

comparaison_rf = pd.DataFrame({
    'MAE_train': [mae_train_lineaire, mae_train_ridge, mae_train_rf],
    'MAE_validation': [mae_sans_mois, mae_ridge, mae_val_rf]
}, index=['Regression_lineaire', 'Ridge', 'Random_Forest'])

comparaison_rf.round(2)

,MAE_train,MAE_validation
Regression_lineaire,181716.82,189013.51
Ridge,181896.07,189582.05
Random_Forest,153840.96,194611.47


le modéle random Forest réduit fortement l'entrainement de train, mais n'améliore pas les prévisions sur Juillet. Premier signal d'un possible surapprentissage. 

Sur la période de validation du 1er au 31 juillet 2025, Random Forest obtient une MAE de **194 611 IDR**, contre 153 841 IDR sur l'entraînement.

L'écart entre ces deux performances est plus marqué que pour les modèles linéaires, ce qui suggère un possible surapprentissage.

Malgré sa capacité à apprendre des relations non linéaires, Random Forest n'améliore pas la précision des prévisions sur juillet.

**La moyenne mobile sur sept jours reste la meilleure méthode**, avec une MAE de 187 223 IDR.

Ces résultats montrent qu'une complexité accrue ne garantit pas de meilleures prévisions, particulièrement avec un historique d'entraînement limité.

In [94]:
periodes_validation = [
    ('2025-05-01', '2025-06-01'),
    ('2025-06-01', '2025-07-01'),
    ('2025-07-01', '2025-08-01')
]

In [95]:
for debut, fin in periodes_validation:
    train_fold = train.loc[train['Date'] < debut]
    val_fold = train.loc[(train['Date'] >= debut) & (train['Date'] < fin)]

    print('Validation :', debut, '→', fin)
    print('Train :', len(train_fold), '| Validation :', len(val_fold))

Validation : 2025-05-01 → 2025-06-01
Train : 336 | Validation : 93
Validation : 2025-06-01 → 2025-07-01
Train : 429 | Validation : 90
Validation : 2025-07-01 → 2025-08-01
Train : 519 | Validation : 93


In [96]:
#Comparaison des modéles 

modeles = {
    'Regression_lineaire': modele_sans_mois,
    'Ridge': modele_ridge,
    'Random_Forest': modele_rf
}

resultats_cv = []

for debut, fin in periodes_validation:
    train_fold = train.loc[train['Date'] < debut]
    val_fold = train.loc[(train['Date'] >= debut) & (train['Date'] < fin)]

    X_train_fold = train_fold[variables_ml_sans_mois]
    y_train_fold = train_fold['CA_net']

    X_val_fold = val_fold[variables_ml_sans_mois]
    y_val_fold = val_fold['CA_net']

    mae_baseline = mean_absolute_error(y_val_fold, val_fold['CA_moyen_7j'])
    resultats_cv.append({'Periode': debut, 'Modele': 'Baseline_7j', 'MAE': mae_baseline})

    for nom, modele in modeles.items():
        modele_fold = clone(modele)
        modele_fold.fit(X_train_fold, y_train_fold)

        predictions = modele_fold.predict(X_val_fold)
        mae = mean_absolute_error(y_val_fold, predictions)

        resultats_cv.append({'Periode': debut, 'Modele': nom, 'MAE': mae})

resultats_cv = pd.DataFrame(resultats_cv)

In [97]:
comparaison_cv = resultats_cv.pivot(index='Periode', columns='Modele', values='MAE')

comparaison_cv.round(2)

Modele,Baseline_7j,Random_Forest,Regression_lineaire,Ridge
Periode,,,,
2025-05-01,197592.06,199821.11,193644.68,193535.15
2025-06-01,224576.01,271445.76,227140.46,227732.41
2025-07-01,187222.71,194611.47,189013.51,189582.05


**Précision méthodologique :** les configurations comparées lors de la validation chronologique ont été partiellement définies à partir d'une première exploration des performances sur juillet.

Les résultats de mai à juillet constituent donc une comparaison rétrospective des méthodes retenues, et non une procédure de sélection entièrement indépendante.

Le jeu de test d'août-septembre a été conservé hors de ce processus et utilisé pour l'évaluation finale.

Une validation chronologique a été réalisée sur trois périodes successives : mai, juin et juillet 2025.

La baseline fondée sur la moyenne mobile de sept jours obtient la meilleure performance sur deux des trois périodes, avec une MAE mensuelle moyenne d'environ **203 130 IDR**.

La régression linéaire présente des performances très proches (203 266 IDR), tandis que Ridge et Random Forest ne démontrent pas d'amélioration globale.

Le mois de juin présente les erreurs les plus importantes pour l'ensemble des méthodes.

**La moyenne mobile sur sept jours est retenue comme méthode de référence privilégiée**, compte tenu de sa simplicité et de ses performances comparables ou supérieures aux modèles entraînés.

Cette sélection reste provisoire et doit être évaluée sur le jeu de test final d'août et septembre 2025.

In [98]:
# Entrainement de notre modéle final

from sklearn.base import clone

modele_final = clone(modele_sans_mois)

X_train_final = train[variables_ml_sans_mois]
y_train_final = train['CA_net']

modele_final.fit(X_train_final, y_train_final)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['CA_J_1','CA_J_7','CA_moyen_7j','Jour_semaine','Outlet']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concate

In [99]:
X_test = test[variables_ml_sans_mois].copy()
y_test = test['CA_net'].copy()

print('X_test :', X_test.shape)
print('y_test :', y_test.shape)

X_test : (183, 5)
y_test : (183,)


In [100]:
y_pred_final = modele_final.predict(X_test)

y_pred_baseline_final = test['CA_moyen_7j']

mae_final_lineaire = mean_absolute_error(y_test, y_pred_final)
mae_final_baseline = mean_absolute_error(y_test, y_pred_baseline_final)

resultats_test = pd.Series({
    'Baseline_7j': mae_final_baseline,
    'Regression_lineaire': mae_final_lineaire
})

resultats_test.round(2)

Baseline_7j            178267.26
Regression_lineaire    172829.86
dtype: float64

L'évaluation finale a été réalisée sur les données du **1er août au 30 septembre 2025**, soit 183 observations réparties entre les trois établissements.

La régression linéaire obtient une MAE de **172 830 IDR**, contre **178 267 IDR** pour la baseline fondée sur la moyenne mobile de sept jours.

Le modèle linéaire améliore ainsi la précision moyenne des prévisions de **3,05 %** sur cette période.

Ce résultat montre qu'une combinaison de variables historiques et calendaires peut apporter un gain prédictif, même si celui-ci reste limité.

La baseline demeure néanmoins une alternative pertinente compte tenu de sa simplicité et de ses performances proches de celles du modèle linéaire.

Ces résultats concernent des prévisions quotidiennes à un jour d'avance et ne garantissent pas une performance équivalente sur d'autres périodes.

In [101]:
resultats_par_shop = test[['Outlet', 'CA_net', 'CA_moyen_7j']].copy()
resultats_par_shop['Prediction_lineaire'] = y_pred_final

mae_test_par_shop = resultats_par_shop.groupby('Outlet').apply(
    lambda groupe: pd.Series({
        'Baseline_7j': mean_absolute_error(groupe['CA_net'], groupe['CA_moyen_7j']),
        'Regression_lineaire': mean_absolute_error(groupe['CA_net'], groupe['Prediction_lineaire'])
    }),
    include_groups=False
)

mae_test_par_shop['Gain_pct'] = (1 - mae_test_par_shop['Regression_lineaire'] / mae_test_par_shop['Baseline_7j']) * 100

mae_test_par_shop.round(2)

,Baseline_7j,Regression_lineaire,Gain_pct
Outlet,,,
SHOP001,214051.52,206233.36,3.65
SHOP002,151701.58,140594.69,7.32
SHOP003,169048.67,171661.52,-1.55


Conclusion générale

L'objectif de cette étude était de prédire le chiffre d'affaires quotidien de trois établissements à partir de leurs performances historiques et des informations calendaires.

Plusieurs méthodes ont été comparées : moyenne mobile, régression linéaire, Ridge et Random Forest.

Sur la période de test du **1er août au 30 septembre 2025**, la régression linéaire obtient une MAE de **172 830 IDR**, contre **178 267 IDR** pour la moyenne mobile sur sept jours, soit une réduction des erreurs de **3,05 %**.

L'amélioration concerne SHOP001 (+3,65 %) et SHOP002 (+7,32 %), tandis que SHOP003 enregistre une légère dégradation (−1,55 %).

**La régression linéaire apporte ainsi un gain prédictif modeste sur la période étudiée**, sans démontrer une supériorité systématique sur la méthode de référence.

Ces résultats concernent des prévisions quotidiennes à un jour d'avance. Ils restent limités par la durée de l'historique disponible et l'absence de certaines variables externes susceptibles d'influencer l'activité.